# Wolf-of-Wallstreet — train on Google Colab (clean v2)

**Every problem we hit is fixed in `scripts/pretrain.py`:** the Binance geo-block (HTTP 451), the 2025 µs-timestamp crash, the RAM out-of-memory (“^C”) during sequence building, the PyTorch `verbose` crash, **and the disk overflow** (no more 41 GB second copy). Plus two model upgrades: **recency-weighting** (the model leans toward the current market regime) and **AMP mixed-precision** (2–3× faster on L4/A100).

### 3 steps
1. **Runtime → Change runtime type → L4 GPU** (best value with compute units; pick **A100** for the fastest finish). Save.
2. Put your **`trading-agent`** folder in Google Drive (*My Drive*) with `scripts/`, `backend/`, `requirements.txt`, **and your `.env`** (it holds your Alpaca keys + the tuned knobs). You can delete `frontend/`, `backend/.venv/`, `.git/`, `node_modules/`, and `training_data/` to keep the upload small — data re-downloads and caches automatically.
3. **Runtime → Run all.** Approve the Drive popup on the first cell.

The checkpoint is written to **`models/trading_lstm_latest.pt` in your Drive** (survives disconnects). Bring it back to your PC's `trading-agent/models/`.

> **You run directly from the Drive folder** (no copy-to-local), so your `.env` + cached data are found automatically and the checkpoint persists. Only the big float16 dataset is written to fast **local** disk (`/content/_mmap_cache`), which is **wiped at the start of every run** so it can never overflow the 112 GB disk.

> **Keep the tab active** during training — idle Colab tabs get reclaimed. The checkpoint is saved on each new best epoch, so a disconnect after that keeps your progress.

### 1) Mount Drive + verify the project
Edit `PROJECT_DIR` if your folder is named/placed differently. This also checks your uploaded `pretrain.py` actually has all the fixes (so you never run a stale copy again).

In [ ]:
PROJECT_DIR = "/content/drive/MyDrive/trading-agent"   # <-- edit if your folder differs

from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir(PROJECT_DIR)
print('Working dir :', os.getcwd())
print('.env present:', os.path.exists('.env'))

# Confirm the uploaded pretrain.py has ALL the fixes (re-upload it if this fails).
src = open('scripts/pretrain.py', encoding='utf-8').read()
need = ['data-api.binance.vision',      # geo-block fix
        '_build_sequences_to_file',     # OOM fix (streaming)
        'make_split_loaders',           # disk fix (no 41 GB copy) + per-symbol split
        '_recency_weights']             # recency weighting
missing = [n for n in need if n not in src]
print('pretrain.py fixes:', '✅ all present' if not missing else f'❌ MISSING {missing} — re-upload scripts/pretrain.py')
assert os.path.exists('.env'), 'No .env in the project folder — upload it (Alpaca keys + tuned knobs).'
assert not missing, 'Old pretrain.py — re-upload the latest scripts/pretrain.py to Drive, then re-run.'

### 2) Install dependencies
Tiny + pure-Python. **No `pandas_ta`, no numpy/pandas pins** — `pretrain.py` has a built-in indicator shim, so Colab's default numpy 2 + pandas stay put (no conflicts). Ignore any ‘fix pandas’ suggestions.

In [ ]:
!pip install -q structlog python-dotenv pydantic pydantic-settings requests scikit-learn
!pip install -q sentence-transformers   # semantic news embeddings (NN_NEWS_EMBED_BACKEND=transformer)
!python -c "import numpy,pandas,torch,sklearn; print('OK | numpy',numpy.__version__,'| pandas',pandas.__version__,'| torch',torch.__version__)"

### 3) Confirm the GPU + disk
The dataset build is CPU/disk-bound; the GPU only engages once training starts. A faster GPU (L4/A100) speeds the **epochs**, not the build.

In [ ]:
import torch, shutil
print('CUDA :', torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print('GPU  :', p.name, f'({p.total_memory/1e9:.0f} GB)')
else:
    print('No GPU — Runtime → Change runtime type → L4 (or T4), then Run all again.')
t, u, f = shutil.disk_usage('/content')
print(f'Disk /content: {f/1e9:.0f} GB free of {t/1e9:.0f} GB  (the build needs ~45 GB)')

### 4) Training knobs (read from your uploaded `.env`)
Already tuned for generalization + risk-adjusted returns, and read from `.env` so they match what you'll run live. **Don't override them here.** `NN_RECENCY_*` is what makes the model emphasise newer data (current regime).

In [ ]:
from dotenv import dotenv_values
cfg = dotenv_values('.env')
print('— model / regularization —')
for k in ['NN_RNN_TYPE','NN_DROPOUT','NN_WEIGHT_DECAY','NN_LABEL_SMOOTHING','NN_AUGMENT_NOISE_STD']:
    print(f'  {k:26s}= {cfg.get(k)}')
print('— recency weighting (newer data emphasised) —')
for k in ['NN_RECENCY_HALFLIFE_YEARS','NN_RECENCY_FLOOR']:
    print(f'  {k:26s}= {cfg.get(k) or "(default 2.0 / 0.25)"}')
print('— reward / sizing (live) —', {k: cfg.get(k) for k in ['NN_KELLY_FRACTION','NN_MIN_EDGE_OVER_FEE','RISK_MAX_POSITION_PCT']})
assert cfg.get('ALPACA_API_KEY'), 'No ALPACA_API_KEY in .env — the 5 stocks would be skipped.'
print('\nALPACA key loaded ✅  — the 5 stocks will train.')

### 5) Train ALL 13 symbols (8 crypto + 5 stocks) in one pass
`--mmap` streams a float16 dataset from local disk (low RAM **and** low disk now — no assembled copy). `--amp` is mixed-precision (faster on L4/A100; harmless on T4). The model + regularization come from your `.env`.

**Tuning the run:**
- `START_YEAR` — more history = better, but longer. **2022** is the safe full run. Set **2024** for a quick ~half-length first pass to confirm the pipeline, or **2020** to include the COVID era for *crypto* (Alpaca stock history is shorter regardless; recency-weighting balances the mix).
- `BATCH_SIZE` — **512** suits L4/A100 with AMP. Drop to **256** on T4 or if the loss looks unstable; raise to **1024** on A100 to go faster.
- Early stopping (`PATIENCE`) ends training once val-loss stops improving, so a high `EPOCHS` is safe.
- For a **seed ensemble** (more robust), run this cell 2–3 times changing `SEED` to 0, 1, 2 — each saves a `..._seed{N}.pt`. (Averaging them live is a follow-up.)

In [ ]:
START_YEAR = 2022      # 2024 = quick first pass | 2020 = include COVID era (crypto)
EPOCHS     = 50        # high is fine — early stopping ends it when val stops improving
PATIENCE   = 8
BATCH_SIZE = 512       # 256 on T4 / 1024 on A100
SEED       = 42        # change to 0,1,2 for ensemble members
NEWS_ALIGN = 0         # 1 = fold SEMANTIC news into training (slower + needs Alpaca news; mainly helps the 5 stocks)
SYMBOLS    = "BTCUSDT ETHUSDT SOLUSDT AAVEUSDT XLMUSDT XRPUSDT ADAUSDT DOGEUSDT SNDK AMD MU AXTI BE"

!PRETRAIN_NEWS_ALIGN={NEWS_ALIGN} python scripts/pretrain.py --mmap --mmap-dir /content/_mmap_cache --amp \
    --start-year {START_YEAR} --epochs {EPOCHS} --patience {PATIENCE} \
    --batch-size {BATCH_SIZE} --seed {SEED} --symbols {SYMBOLS}

### 6) Verify the checkpoint
Saved to `models/` in your Drive. Download it and drop it into your PC's `trading-agent/models/`.

In [ ]:
import os, glob
for ckpt in ['models/trading_lstm_latest.pt'] + sorted(glob.glob('models/trading_lstm_latest_seed*.pt')):
    if os.path.exists(ckpt):
        print(f'✓ {ckpt}  ({os.path.getsize(ckpt)/1e6:.1f} MB)  — saved in your Drive.')
if not os.path.exists('models/trading_lstm_latest.pt'):
    print('No default checkpoint yet — did training finish (or did you only run non-default seeds)?')

### 7) Out-of-sample backtest — the measuring stick
Runs the trained model over each symbol's **held-out tail** with **fees + slippage**, reporting Sharpe / Sortino / max-DD / hit-rate per symbol + aggregate. **This is how you judge whether a change actually helped** — compare these numbers between runs, not the validation loss. Sharpe > 1 with positive return *after costs* across most symbols = a real edge.

In [ ]:
# Out-of-sample backtest of the model you just trained (uses the cached data).
!python scripts/backtest.py --checkpoint models/trading_lstm_latest.pt \
    --start-year {START_YEAR} --skip-download \
    --fee-bps 10 --slippage-bps 5 --min-confidence 0.45 --min-edge 0.05

In [ ]:
# Same model, NOW with the Cycle-5 execution layer (vol-targeted sizing + ATR
# trailing stop + breakeven + partial scale-out). Compare these metrics to the
# plain backtest above — that's how you judge whether the execution layer helped.
!python scripts/backtest.py --checkpoint models/trading_lstm_latest.pt \
    --start-year {START_YEAR} --skip-download --exec \
    --fee-bps 10 --slippage-bps 5 --min-confidence 0.45 --min-edge 0.05 \
    --stop-atr 2.0 --trail-atr 3.0 --breakeven-atr 1.0 --tp1-atr 2.0 --scale-out 0.5